# RAG Pipeline for Product Description Q&A

This notebook demonstrates a complete **Retrieval-Augmented Generation (RAG) pipeline** that answers natural-language questions over a corpus of ~500 synthetic product-description documents.


**Key features:**
- 🧱 Four chunking strategies: `fixed_overlap`, `sentence`, `paragraph`, `recursive`
- 🔍 Hybrid dense (cosine) + sparse (BM25) retrieval with Reciprocal Rank Fusion
- ✂️ Token-budget–aware context assembly via `tiktoken`
- 🚫 100% local — no API keys, no cloud services


## 0. Setup

In [ ]:
import logging
import os
import sys
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

# Add project root to path (in case running from a subdirectory)
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

print('Setup complete')
print(f'   Python: {sys.version.split()[0]}')
print(f'   NumPy:  {np.__version__}')
print(f'   Pandas: {pd.__version__}')

## 1. Corpus Generation

Generate 500 synthetic product documents across 10 categories using `faker` + handcrafted templates.

In [ ]:
from rag.corpus import CorpusGenerator
from rag.models import VALID_CATEGORIES

t0 = time.perf_counter()
generator = CorpusGenerator(seed=42)
docs = generator.generate(n=500)
elapsed = time.perf_counter() - t0

print(f'✅ Generated {len(docs)} documents in {elapsed:.2f}s')
print(f'\nSample document:')
sample = docs[0]
print(f'  doc_id    : {sample.doc_id}')
print(f'  name      : {sample.name}')
print(f'  category  : {sample.category}')
print(f'  features  : {sample.features[:2]} ...')
print(f'  specs     : {sample.specs}')
print(f'  created_at: {sample.created_at}')
print(f'  word count: {len(sample.description.split())} words')
print(f'\nDescription preview:')
print(f'  "{sample.description[:200]}..."')

In [ ]:
# Category distribution
cat_counts = Counter(d.category for d in docs)
df_cats = pd.DataFrame([
    {'Category': cat, 'Count': cnt, 'Share (%)': f'{cnt/len(docs)*100:.1f}'}
    for cat, cnt in sorted(cat_counts.items())
])
print('Category Distribution:')
print(df_cats.to_string(index=False))

# Word count stats
word_counts = [len(d.description.split()) for d in docs]
print(f'\nDescription word count stats:')
print(f'  min={min(word_counts)}, max={max(word_counts)}, mean={np.mean(word_counts):.0f}, median={np.median(word_counts):.0f}')

## 2. Document Chunking

Split documents into `Chunk` objects using all four strategies and compare their statistics.

In [ ]:
from rag.chunker import DocumentChunker

strategies = ['fixed_overlap', 'sentence', 'paragraph', 'recursive']
strategy_stats = []

for strategy in strategies:
    t0 = time.perf_counter()
    chunker = DocumentChunker(strategy=strategy, chunk_size=128, overlap=25)
    all_chunks = chunker.chunk_batch(docs)
    elapsed = time.perf_counter() - t0
    
    token_counts = [c.token_count for c in all_chunks]
    strategy_stats.append({
        'Strategy': strategy,
        'Total Chunks': len(all_chunks),
        'Chunks/Doc': f'{len(all_chunks)/len(docs):.1f}',
        'Min Tokens': min(token_counts),
        'Max Tokens': max(token_counts),
        'Mean Tokens': f'{np.mean(token_counts):.0f}',
        'Time (s)': f'{elapsed:.2f}',
    })

df_stats = pd.DataFrame(strategy_stats)
print('Chunking Strategy Comparison:')
print(df_stats.to_string(index=False))

In [ ]:
# Show example chunks from fixed_overlap
chunker = DocumentChunker(strategy='fixed_overlap', chunk_size=128, overlap=25)
example_doc = docs[0]
example_chunks = chunker.chunk(example_doc)

print(f'Document: {example_doc.name} ({example_doc.category})')
print(f'Produced {len(example_chunks)} chunks\n')
for ch in example_chunks[:3]:
    print(f'--- Chunk {ch.chunk_index} ({ch.token_count} tokens) [{ch.chunk_id}] ---')
    print(ch.text[:250])
    print()

## 3. Embedding

Encode chunks into dense vectors. We use `all-MiniLM-L6-v2` (384-dim) if available, otherwise `MockEmbedder` (64-dim).

In [ ]:
from rag.embeddings import MockEmbedder, SentenceTransformerEmbedder, get_embedder

# Use MockEmbedder by default; set USE_REAL_EMBEDDER=True to use all-MiniLM-L6-v2
USE_REAL_EMBEDDER = os.getenv('RAG_USE_MOCK_EMBEDDER', 'true').lower() != 'true'

if USE_REAL_EMBEDDER:
    try:
        embedder = SentenceTransformerEmbedder()
        print(f'✅ Using SentenceTransformerEmbedder: all-MiniLM-L6-v2 (dim={embedder.dim})')
    except Exception as e:
        print(f'⚠️  Could not load real model ({e}). Falling back to MockEmbedder.')
        embedder = MockEmbedder(dim=64)
else:
    embedder = MockEmbedder(dim=64)
    print(f'ℹ️  Using MockEmbedder (dim={embedder.dim}) — semantic quality not guaranteed.')
    print('   Set RAG_USE_MOCK_EMBEDDER=false to use the real model.')

# Encode a single query
t0 = time.perf_counter()
test_vec = embedder.encode('wireless noise-cancelling headphones')
single_ms = (time.perf_counter() - t0) * 1000
print(f'\nSingle encode: shape={test_vec.shape}, dtype={test_vec.dtype}, L2-norm={np.linalg.norm(test_vec):.6f}')
print(f'Single encode time: {single_ms:.1f}ms')

In [ ]:
# Batch encode all chunks
chunker = DocumentChunker(strategy='fixed_overlap', chunk_size=128, overlap=25)
all_chunks = chunker.chunk_batch(docs)
chunk_texts = [c.text for c in all_chunks]

print(f'Encoding {len(all_chunks)} chunks...')
t0 = time.perf_counter()
all_vectors = embedder.encode_batch(chunk_texts, batch_size=64)
elapsed = time.perf_counter() - t0

print(f'✅ Batch encode complete:')
print(f'   Shape  : {all_vectors.shape}')
print(f'   dtype  : {all_vectors.dtype}')
print(f'   Time   : {elapsed:.2f}s ({elapsed/len(all_chunks)*1000:.1f}ms/chunk)')
print(f'   L2-norms: min={np.linalg.norm(all_vectors, axis=1).min():.6f}, max={np.linalg.norm(all_vectors, axis=1).max():.6f}')

## 4. Indexing

Add all chunks and vectors to the `VectorStore`. The store builds a dense NumPy index and a BM25 inverted index.

In [ ]:
from rag.vector_store import VectorStore

t0 = time.perf_counter()
store = VectorStore(embedding_dim=embedder.dim)
store.add(all_chunks, all_vectors)
elapsed = time.perf_counter() - t0

print(f'✅ Indexed {len(store)} chunks in {elapsed:.2f}s')
print(f'   Dense index shape : ({len(store)}, {embedder.dim})')
print(f'   Memory estimate   : {len(store) * embedder.dim * 4 / 1024 / 1024:.1f} MB (float32 dense)')
print(f'\nCategories in store:')
for cat in store.list_metadata_values('category'):
    print(f'   {cat}')

## 5. Query Demonstration

Run 3+ example queries and display ranked search results.

In [ ]:
from rag.context_manager import ContextManager

context_manager = ContextManager(budget_tokens=1024, dedup_threshold=0.85)


def run_query(query: str, top_k: int = 5, filter_metadata=None, mode='hybrid'):
    """Run a complete query through the RAG pipeline and display results."""
    print(f'\n{"="*60}')
    print(f'QUERY: {query!r}')
    if filter_metadata:
        print(f'FILTER: {filter_metadata}')
    print(f'MODE: {mode} | TOP-K: {top_k}')
    print('='*60)
    
    t0 = time.perf_counter()
    q_vec = embedder.encode(query)
    results = store.search(q_vec, query, top_k=top_k, filter_metadata=filter_metadata, mode=mode)
    bundle = context_manager.build_context(results, query)
    elapsed = (time.perf_counter() - t0) * 1000
    
    print(f'Retrieved: {len(results)} results → {len(bundle.included_chunk_ids)} included after dedup')
    print(f'Tokens used: {bundle.total_tokens_used}/{bundle.budget_tokens} | Latency: {elapsed:.0f}ms')
    print()
    
    df = pd.DataFrame([
        {
            'Rank': i+1,
            'Chunk ID': r.chunk.chunk_id,
            'Category': r.chunk.metadata.get('category', '?'),
            'Dense': f'{r.dense_score:.3f}',
            'Sparse': f'{r.sparse_score:.2f}',
            'RRF': f'{r.rrf_score:.4f}',
            'Tokens': r.chunk.token_count,
            'Text Preview': r.chunk.text[:80] + '...'
        }
        for i, r in enumerate(results)
    ])
    print(df.to_string(index=False))
    return bundle

In [ ]:
# Query 1: Electronics — audio
bundle1 = run_query('wireless noise cancelling headphones with long battery life', top_k=5)

In [ ]:
# Query 2: Furniture — ergonomics
bundle2 = run_query('ergonomic office chair with lumbar support', top_k=5)

In [ ]:
# Query 3: Sports — home gym equipment
bundle3 = run_query('lightweight adjustable dumbbells for home gym', top_k=5)

In [ ]:
# Query 4: Category-filtered search (electronics only)
bundle4 = run_query(
    'portable bluetooth speaker waterproof',
    top_k=5,
    filter_metadata={'category': 'electronics'}
)

In [ ]:
# Query 5: Dense-only vs Sparse-only vs Hybrid comparison
query5 = 'non-stick induction compatible cookware'
q5_vec = embedder.encode(query5)

print(f'\nRetrieval mode comparison for: {query5!r}\n')
for mode in ['dense', 'sparse', 'hybrid']:
    results = store.search(q5_vec, query5, top_k=3, mode=mode)
    print(f'{mode.upper():8s}: {[r.chunk.chunk_id for r in results]}')

## 6. Context Assembly

Show the assembled context string with numbered citations, deduplication, and token budget.

In [ ]:
print('ASSEMBLED CONTEXT STRING (Query 1 — headphones):')
print('─' * 60)
print(bundle1.context_str)
print('─' * 60)
print(f'Total tokens: {bundle1.total_tokens_used} / {bundle1.budget_tokens}')
print(f'Included chunks: {bundle1.included_chunk_ids}')

## 7. Mock LLM

Pass the context bundle to `MockLLM` to demonstrate the complete prompt assembly.

In [ ]:
from rag.mock_llm import MockLLM

llm = MockLLM(verbose=True)
query = 'wireless noise cancelling headphones with long battery life'
answer = llm.answer(query, bundle1)

print('\n' + '='*60)
print('MockLLM Answer:')
print('='*60)
print(answer)

## 8. Performance Benchmarks

Measure end-to-end latency for the query path.

In [ ]:
import statistics

queries = [
    'wireless headphones battery life',
    'ergonomic chair lumbar support',
    'stainless steel cookware induction',
    'lightweight running jacket waterproof',
    'bluetooth speaker portable waterproof',
]

latencies = []
for q in queries:
    t0 = time.perf_counter()
    qv = embedder.encode(q)
    results = store.search(qv, q, top_k=20)
    bundle = ContextManager(budget_tokens=2048).build_context(results, q)
    latencies.append((time.perf_counter() - t0) * 1000)

print('Query → Context Latency (ms):')
for q, lat in zip(queries, latencies):
    print(f'  {lat:5.0f}ms  {q!r}')
print(f'\n  Mean   : {statistics.mean(latencies):.0f}ms')
print(f'  Median : {statistics.median(latencies):.0f}ms')
print(f'  Max    : {max(latencies):.0f}ms')

## 9. Design Questions

### Q1: Why is `fixed_overlap` the default chunking strategy for short product descriptions?

**Answer:**

Product descriptions are short (50–300 words), structurally uniform, and contain dense keyword-bearing phrases. The `fixed_overlap` strategy offers three concrete advantages over the alternatives in this context:

1. **Boundary phrase preservation.** A 20% token overlap (25 tokens for a 128-token chunk) ensures that semantically important phrases at split boundaries appear in at least one chunk in full. Without overlap, the phrase *"noise-cancelling headphones with 30-hour battery"* could be split into *"noise-cancelling headphones"* and *"with 30-hour battery"* — weakening both chunks' retrieval signal.

2. **Predictable chunk size keeps embedding batch sizes uniform.** Sentence-based chunking produces variable-length batches that cause memory spikes. Fixed-size chunks produce consistent token counts, which is important for batched inference on CPU-only hardware where memory is a hard constraint.

3. **Stable retrieval latency.** The in-memory NumPy cosine search is O(N·D). When chunk counts are predictable, latency is predictable. Recursive or paragraph strategies can produce wildly different chunk counts per document depending on document structure, introducing query latency variance.

**Sentence strategy would be preferred** if documents had multiple structurally distinct paragraphs with clear topic shifts (e.g., long-form reviews). **Paragraph strategy** suits structured documents with explicit paragraph breaks. For this corpus — which is template-generated with a single narrative description — `fixed_overlap` dominates.

---

### Q2: Explain the Reciprocal Rank Fusion (RRF) formula and why it was chosen over alternatives.

**Answer:**

**The formula:**

```
RRF(d) = 1/(k + rank_dense(d)) + 1/(k + rank_sparse(d))
```

where `k=60` is a smoothing constant that prevents very highly-ranked documents from dominating, and `rank_dense`/`rank_sparse` are the 1-indexed positions of document `d` in each ranked list.

**Why RRF over alternatives:**

| Alternative | Problem |
|---|---|
| **Linear score combination** (`α·dense + β·sparse`) | Dense scores ∈ [0, 1]; BM25 scores ∈ [0, ∞). Naive sum is dominated by BM25. Normalisation is non-trivial and corpus-dependent. |
| **Dense-only** | Fails on exact-match queries (SKU codes, product names). Embedding models struggle with rare terminology. |
| **Sparse-only** | Fails on semantic paraphrase queries ("quiet headphones" vs. "noise-cancelling headphones"). |
| **Learned fusion** | Requires labelled query-document pairs to train. Not available for a synthetic corpus. |

**RRF's advantages:**
1. **Scale-invariant.** Operates on ranks, not raw scores. No normalisation needed regardless of score magnitude differences between dense and sparse retrievers.
2. **Parameter-robust.** `k=60` is empirically well-validated across many IR tasks. The formula is not sensitive to small changes in k.
3. **Principled.** Documents ranked highly by *both* systems receive multiplicatively higher scores than those ranked highly by only one.

**The `k=60` constant** dampens the advantage of the top-ranked document. Without `k`, a rank-1 document gets score 1.0 and rank-2 gets 0.5 — too extreme. With `k=60`, rank-1 gets 1/61 ≈ 0.0164 and rank-2 gets 1/62 ≈ 0.0161 — more gradual and stable.

---

### Q3: When would hybrid retrieval hurt performance compared to single-mode retrieval?

**Answer:**

Hybrid retrieval can degrade precision compared to either retriever alone in four scenarios:

1. **Exact-match queries where sparse is superior.** For queries like `"Sony WH-1000XM5"`, BM25 ranks the exact-match document first with near-perfect precision. Dense retrieval returns semantically similar but factually different documents (other headphone models). After RRF fusion, the correct document's rank can fall because high-scoring dense candidates dilute BM25's rank-1 advantage.

2. **Domain-specific terminology outside the embedding model's training distribution.** If the corpus uses proprietary spec codes or highly specialised jargon that was underrepresented in `all-MiniLM-L6-v2`'s training data, dense retrieval produces low-quality rankings. RRF fusion then degrades the high-quality sparse-only result by blending in noise.

3. **Short, high-precision queries over a large corpus.** A two-word query with a clear exact match (e.g., `"bluetooth 5.3"`) is handled perfectly by BM25 but poorly by dense retrieval, which returns dozens of semantically adjacent documents. After fusion, the correct document's RRF score may be outweighed by the aggregate density of semantically adjacent matches.

4. **Degenerate dense retrieval (mock or poorly trained embedder).** When dense vectors are random (as with `MockEmbedder`), dense rankings are noise. RRF dilutes the BM25 signal with this noise, reducing overall performance below sparse-only.

**Practical mitigation:** The `VectorStore.search(mode=...)` parameter allows callers to select `"sparse"`, `"dense"`, or `"hybrid"` per query. A query classifier (e.g., detecting SKU patterns or very short queries) can route to the appropriate mode automatically.

## 10. CRUD Operations Demo

Show add, delete, and get operations on the VectorStore.

In [ ]:
# Demonstrate CRUD
print(f'Store size before delete: {len(store)}')

# Delete a specific chunk
target_id = all_chunks[0].chunk_id
deleted = store.delete([target_id])
print(f'Deleted {deleted} chunk(s): {target_id}')
print(f'Store size after delete: {len(store)}')
print(f'get({target_id!r}): {store.get(target_id)}')

# Re-add the chunk
store.add([all_chunks[0]], all_vectors[[0]])
print(f'\nRe-added chunk. Store size: {len(store)}')
print(f'get({target_id!r}): {store.get(target_id).chunk_id}')